# Hard Spark Interview Prep — Senior Engineer (6-12 Years)

**Difficulty:** Hard
**Audience:** Senior Data Engineers, Spark Architects
**Dataset:** NYC Cabs.csv + synthetic data via `spark.range()`

| # | Topic |
|---|-------|
| 1 | Large Table Joins |
| 2 | Data Skew Handling |
| 3 | Small Files Problem |
| 4 | Executor Memory / OOM |
| 5 | Shuffle Spill Analysis |
| 6 | Dynamic Partitioning |
| 7 | Delta Lake Optimization |
| 8 | Z-Ordering |

Each topic follows cells A→M: Business Scenario, Data Setup, Problem, Bad Code, Interview Questions, Logical Plan, Physical Plan, Spark UI Guide, Exercise, Benchmark, Best Practices, Follow-ups, Full Solution.

In [ ]:
import os, sys, time
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (col, lit, rand, when, desc, asc, count,
                                   sum as spark_sum, avg, concat, expr,
                                   year, month, dayofmonth, row_number, lag,
                                   broadcast)
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (SparkSession.builder
    .appName("SparkInterviewPrep-Hard")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

CABS_PATH = r"C:\Users\PS\Documents\Python-Exp\RawData\Cabs.csv"
cabs = spark.read.csv(CABS_PATH, header=True, inferSchema=True)
cabs.cache()
print(f"Cabs rows: {cabs.count()}, columns: {len(cabs.columns)}")
print("Columns:", cabs.columns)

## Topic 1 — Large Table Joins

### Business Scenario

| Aspect | Detail |
|--------|--------|
| Symptom | Nightly join of two 50M-row fact tables takes 45 min; Stage 3 dominates |
| Impact | SLA breach; downstream reporting delayed by 6 hours |
| Goal | Reduce join stage from 45 min to < 5 min without adding hardware |

**Dataset:** Two synthetic tables (`trips` and `fares`) joined on `trip_id`; `cabs` used as a dimension.

In [ ]:
# ── Topic 1 — Data Setup ──────────────────────────────────────────────────────
trips = (spark.range(500_000)
    .withColumnRenamed("id", "trip_id")
    .withColumn("cab_number", (col("trip_id") % 8970).cast("long"))
    .withColumn("distance_miles", (rand(seed=1) * 30).cast("double"))
    .withColumn("trip_date", expr("date_add(date'2024-01-01', cast(trip_id % 365 as int))")))

fares = (spark.range(500_000)
    .withColumnRenamed("id", "trip_id")
    .withColumn("fare_amount", (rand(seed=2) * 100 + 5).cast("double"))
    .withColumn("tip_amount", (rand(seed=3) * 20).cast("double")))

print("trips:", trips.count(), "| fares:", fares.count())
trips.printSchema()

### Problem Statement — SortMergeJoin with No Optimization

**Production symptoms:**
- Stage 3 (join stage) has **Exchange** nodes on both sides → full shuffle of both tables
- 200 shuffle partitions each writing tiny files → excessive task overhead
- SortMergeJoin sorts 2 × 50M rows even though the join key is monotonically increasing
- No predicate pushdown → full scan of both tables before join

**Anti-pattern:** Joining two large tables with default settings, no bucketing, no partition pruning.

In [ ]:
# ── Topic 1 — BAD PATTERN ────────────────────────────────────────────────────
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force SortMergeJoin

# BAD: No optimization — both sides shuffle + sort
result_bad = trips.join(fares, on="trip_id", how="inner")
print("=== BAD: SortMergeJoin plan ===")
result_bad.explain()
# Observe: two Exchange hashpartitioning nodes + Sort on both sides

### Interview Questions — Topic 1: Large Table Joins

1. A join stage takes 40 minutes while all other stages complete in seconds. How do you diagnose whether it is a join strategy problem, a skew problem, or a partition count problem?

2. Explain the difference between `BroadcastHashJoin`, `SortMergeJoin`, `ShuffledHashJoin`, `BroadcastNestedLoopJoin`, and `CartesianProduct`. When does Spark choose each?

3. You have two 10 GB tables joining on `user_id`. One table has 1 billion rows, the other has 500 million. Neither fits in broadcast memory. What are all the options you have to optimize this join?

4. A colleague adds `broadcast(smallDf)` hint but Spark ignores it and still uses SortMergeJoin. Name three reasons this could happen.

5. Explain bucket joins. What conditions must be met for Spark to completely eliminate the Exchange node on both sides of a bucketed join?

6. How does AQE's runtime join conversion work? At what point in execution does the strategy switch, and what triggers it?

7. You bucketed a table with `bucketBy(200, "user_id")`. A new team joins it with a table bucketed with `bucketBy(100, "user_id")`. What happens? How do you fix it?

8. Explain `spark.sql.join.preferSortMergeJoin`. When would you set it to false, and what are the risks?

### Expected Logical Plan Discussion — Large Table Joins

**Catalyst optimization rules that fire:**
- `EliminateSorts` — removes redundant sorts if data already ordered
- `ReorderJoin` — reorders joins to minimize intermediate result size
- `PushDownPredicates` — pushes filters below joins to reduce input
- `CostBasedJoinReorder` (if CBO enabled) — uses statistics to pick join order

**Key logical nodes:**
```
Join Inner, (trip_id = trip_id)
:- Relation trips [trip_id, cab_number, distance_miles, trip_date]
+- Relation fares [trip_id, fare_amount, tip_amount]
```

With CBO (`ANALYZE TABLE` + `spark.sql.cbo.enabled=true`), Catalyst knows row counts and can pick BHJ if one side is small enough after filter pushdown.

### Expected Physical Plan Discussion — Large Table Joins

**AQE OFF (bad pattern):**
```
SortMergeJoin [trip_id], [trip_id], Inner
:- Sort [trip_id ASC]
:  +- Exchange hashpartitioning(trip_id, 200)
:     +- Scan trips
+- Sort [trip_id ASC]
   +- Exchange hashpartitioning(trip_id, 200)
      +- Scan fares
```

**AQE ON with BHJ (good):**
```
AdaptiveSparkPlan isFinalPlan=true
+- BroadcastHashJoin [trip_id], [trip_id], Inner, BuildRight
   :- Scan trips
   +- BroadcastExchange HashedRelationBroadcastMode(trip_id)
      +- Scan fares (after runtime size measurement)
```

**Bucketed join (best):**
```
SortMergeJoin [trip_id], [trip_id], Inner
:- Sort [trip_id ASC]       ← no Exchange!
:  +- FileScan BucketedScan trips (bucket 0..199 by trip_id)
+- Sort [trip_id ASC]       ← no Exchange!
   +- FileScan BucketedScan fares (bucket 0..199 by trip_id)
```
Key signal: **`Exchange` disappears** when bucketing is aligned.

### Spark UI Investigation Guide — Large Table Joins

**Step 1 — SQL tab → DAG visualization**
- Find the job triggered by your join action
- Look for orange `Exchange` diamonds — each represents a full shuffle
- Count: 2 Exchange nodes = SortMergeJoin; 1 = ShuffledHashJoin; 0 = BroadcastHashJoin or BucketedScan

**Step 2 — Stages tab → identify the join stage**
- Sort stages by "Duration" descending
- Click the slowest stage → examine "Shuffle Read" and "Shuffle Write" sizes
- If Shuffle Write > 1 GB, consider bucketing or broadcast

**Step 3 — Tasks tab within the join stage**
- Look at task duration distribution: min/median/max
- Large max:median ratio (> 5×) = skew, not a join strategy problem
- Uniform but slow tasks = wrong join strategy or partition count

**Step 4 — SQL tab → plan detail**
- Expand the join node: confirm `SortMergeJoin` vs `BroadcastHashJoin`
- Check `numOutputRows` and `numPartitions` metrics
- With AQE: look for `isFinalPlan=true` on the `AdaptiveSparkPlan` node

In [ ]:
# ── Topic 1 — Optimization Exercise (TODO) ───────────────────────────────────
# TODO 1: Re-enable AQE and set a 50 MB broadcast threshold.
#          Run the join and confirm plan shows BroadcastHashJoin.
# TODO 2: Try explicit broadcast() hint and verify Exchange disappears.
# TODO 3: (Advanced) Write both tables as bucketed Parquet (8 buckets, sort by trip_id),
#          re-read, join, and confirm both Exchange nodes are gone from the plan.

# YOUR CODE HERE
spark.conf.set("spark.sql.adaptive.enabled", "true")
# ...


In [ ]:
# ── Topic 1 — Benchmark ──────────────────────────────────────────────────────
def bench(label, df, n=3):
    times = []
    for _ in range(n):
        t = time.time()
        df.count()
        times.append(time.time() - t)
    avg_t = sum(times) / n
    print(f"{label}: avg {avg_t:.2f}s over {n} runs")

spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
bad_join = trips.join(fares, on="trip_id")

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(50 * 1024 * 1024))
good_join = trips.join(fares, on="trip_id")

bench("SortMergeJoin (AQE off, no broadcast)", bad_join)
bench("AQE + broadcast threshold 50MB", good_join)

### Production Best Practices — Large Table Joins

1. **Profile before optimizing** — check Spark UI SQL tab for Exchange node count and Shuffle Write bytes before changing any config.
2. **Prefer AQE runtime broadcast** over static `broadcast()` hints — AQE measures actual post-filter sizes; hints use pre-filter estimates.
3. **Bucket at write time, not read time** — `bucketBy` is only effective when both sides use the same bucket count and join key.
4. **Co-locate dimension tables** — keep slowly changing dimensions small enough to broadcast (< 200 MB after filter).
5. **Avoid cross-joins accidentally** — a missing join condition silently produces a `CartesianProduct`; always check `.explain()`.
6. **Set `spark.sql.cbo.enabled=true`** and run `ANALYZE TABLE` on Hive tables — CBO lets Catalyst pick join order and strategy using real row counts.
7. **Monitor Shuffle Read Bytes in Stages tab** — if it grows proportionally with data volume after optimization, the join strategy has not changed.

### Follow-up Questions — Topic 1

1. You've bucketed both tables with 200 buckets, but the join still shows an `Exchange` node for one side. What are the possible causes?
2. How does `spark.sql.adaptive.localShuffleReader.enabled` reduce network I/O after a broadcast join?
3. A SortMergeJoin is running faster than your bucketed join. What could explain this?
4. Explain how `spark.sql.autoBroadcastJoinThreshold` interacts with AQE's `spark.sql.adaptive.autoBroadcastJoinThreshold`. Which takes precedence?
5. When would you deliberately choose SortMergeJoin over BroadcastHashJoin even when both sides fit in memory?

In [ ]:
# ── Topic 1 — Full Optimized Solution ───────────────────────────────────────
# OPTIMIZED: Re-enable AQE; set broadcast threshold so Spark can switch at runtime
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# OPTIMIZED: 50 MB threshold — fares table is ~40 MB after projection
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(50 * 1024 * 1024))

# OPTIMIZED: Project only needed columns before join to reduce shuffle size
trips_slim = trips.select("trip_id", "cab_number", "distance_miles")
fares_slim = fares.select("trip_id", "fare_amount")

# OPTIMIZED: Explicit broadcast hint as a safety net alongside AQE
result_opt = trips_slim.join(broadcast(fares_slim), on="trip_id", how="inner")
print("=== OPTIMIZED: BroadcastHashJoin plan ===")
result_opt.explain()
# Verify: BroadcastHashJoin + BroadcastExchange; no Exchange hashpartitioning

result_opt.groupBy("cab_number").agg(
    spark_sum("fare_amount").alias("total_revenue"),
    spark_sum("distance_miles").alias("total_miles")
).orderBy(desc("total_revenue")).show(5)

## Topic 2 — Data Skew Handling

### Business Scenario

| Aspect | Detail |
|--------|--------|
| Symptom | One task in the join stage takes 60 min; all others finish in 30 sec |
| Impact | The entire stage waits for the straggler; cluster utilization: 1/200 at 98% of runtime |
| Goal | Distribute the hot-key load across multiple tasks |

**Dataset:** `orders_skewed` with 80% of rows on `vehicle_type="Yellow Taxi"` joined to `cabs`.

In [ ]:
# ── Topic 2 — Data Setup ──────────────────────────────────────────────────────
# Simulate skewed fact table: 80% rows on vehicle_type_id=0 (Yellow Taxi)
orders_skewed = (spark.range(200_000)
    .withColumn("vehicle_type_id",
        when(rand(seed=10) < 0.8, lit(0))
        .otherwise((rand(seed=11) * 9 + 1).cast("long")))
    .withColumn("fare", (rand(seed=12) * 80 + 5).cast("double")))

# Small dimension
vehicle_types = spark.createDataFrame(
    [(i, t) for i, t in enumerate(
        ["Yellow Taxi","Green Cab","Black Car","Livery","FHV",
         "Commuter Van","Para-Transit","Ambulette","Limo","Rideshare"])],
    ["vehicle_type_id","vehicle_type_name"])

print("=== Key distribution (skew evidence) ===")
orders_skewed.groupBy("vehicle_type_id").count().orderBy(desc("count")).show()

### Problem Statement — Hot-Key Straggler

**Symptoms in Spark UI:**
- Stages tab: task duration chart shows 1 bar 100× taller than all others
- "Max Task Duration" >> "Median Task Duration"
- "Shuffle Read Size / Records" column: one task reads 10× more bytes

**Why skew is dangerous:**
- The stage does not complete until the last task finishes
- Speculative execution cannot help (the task is not slow due to a bad node — it has more data)
- Memory pressure on the executor holding the hot key risks OOM

In [ ]:
# ── Topic 2 — BAD PATTERN ────────────────────────────────────────────────────
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force SMJ

# BAD: Plain join — hot key lands on one partition → straggler
bad_result = orders_skewed.join(vehicle_types, on="vehicle_type_id", how="inner")
print("=== BAD: plain SMJ on skewed key ===")
bad_result.explain()
# One partition will have ~160,000 rows; all others have ~2,000 rows each

### Interview Questions — Topic 2: Data Skew Handling

1. How do you distinguish a data skew problem from a bad join strategy problem when looking at the Spark UI? What exact metrics and tabs do you check?

2. Explain the salting technique for skew joins. Walk through the exact steps: what you add to the fact table, what you do to the dimension table, and why.

3. What are `spark.sql.adaptive.skewJoin.skewedPartitionFactor` and `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes`? How does AQE use both to identify a skewed partition?

4. Your data has a skewed key that also happens to be the partition key on disk. Does AQE's skewJoin fix partition-level skew, or only shuffle-level skew?

5. Salting increases the number of output rows on the dimension side. For a 10-salt approach on a 1 GB dimension table, what is the new dimension size? Is this always acceptable?

6. When would you choose the "isolate hot key" pattern (filter + union) over salting? What are the trade-offs?

7. A job has skew on 3 different join keys in 3 different stages. You enable AQE skewJoin. Explain exactly what Spark does differently for each skewed partition at runtime.

8. After salting, your results have duplicates. What went wrong, and how do you fix it?

### Expected Logical Plan Discussion — Data Skew

**Without optimization:**
```
Join Inner, (vehicle_type_id = vehicle_type_id)
:- Relation orders_skewed [vehicle_type_id, fare]
+- Relation vehicle_types [vehicle_type_id, vehicle_type_name]
```

**With salting (logical view):**
```
Project [vehicle_type_id, fare, vehicle_type_name]
+- Join Inner, (salt_key = salt_key AND vehicle_type_id = vehicle_type_id)
   :- Project [vehicle_type_id, fare, (rand() * 10).cast(int) AS salt]
   :  +- Relation orders_skewed
   +- Project [vehicle_type_id, vehicle_type_name, explode(array(0..9)) AS salt_key]
      +- Relation vehicle_types
```

Catalyst sees through the `explode` + `rand()` and does NOT push predicates across it (correct — each salt bucket must match independently).

### Expected Physical Plan Discussion — Data Skew

**AQE skewJoin enabled (ideal cluster scenario):**
```
AdaptiveSparkPlan isFinalPlan=true
+- SortMergeJoin [vehicle_type_id], [vehicle_type_id], Inner
   :- AQEShuffleRead isSkewed   ← split hot partition into sub-tasks
   :  +- Exchange hashpartitioning(vehicle_type_id, 200)
   :     +- Scan orders_skewed
   +- AQEShuffleRead
      +- Exchange hashpartitioning(vehicle_type_id, 200)
         +- Scan vehicle_types
```

**Salting physical plan:**
```
SortMergeJoin [vehicle_type_id#salt], [vehicle_type_id#salt], Inner
:- Exchange hashpartitioning(vehicle_type_id, salt, 200)
:  +- Project [vehicle_type_id, fare, (rand()*10).cast(int) AS salt]
+- Exchange hashpartitioning(vehicle_type_id, salt, 200)
   +- Generate explode([0,1,2,3,4,5,6,7,8,9])
      +- Scan vehicle_types
```

### Spark UI Investigation Guide — Data Skew

**Step 1 — Stages tab → Tasks column**
- Click the slow stage → scroll to "Tasks" section
- Sort by "Duration" descending
- If Task 0 took 60 min and Task 1-199 took 30 sec each → skew confirmed

**Step 2 — Stages tab → Shuffle Read Size column**
- Click "Show Additional Metrics" → enable "Shuffle Read Size / Records"
- One task reading 5 GB while others read 25 MB = hot-key skew

**Step 3 — SQL tab → SortMergeJoin node metrics**
- Expand the SMJ operator in the DAG
- `numOutputRows` per task: one task shows 10× more than median

**Step 4 — Confirm with data distribution**
- Add `df.groupBy(joinKey).count().orderBy(desc("count")).show(10)` before the join
- If top key > 5× median count → salting or AQE skewJoin needed

**Step 5 — After AQE skewJoin**
- SQL tab: look for `AQEShuffleRead` node with `isSkewed=true` metric
- Stages tab: task durations should now be uniform

In [ ]:
# ── Topic 2 — Optimization Exercise (TODO) ───────────────────────────────────
# TODO 1: Enable AQE skewJoin and lower the threshold to trigger on local data.
#          Confirm the plan shows AQEShuffleRead with skew handling.
# TODO 2: Implement salting with 4 salt buckets. Join on (vehicle_type_id, salt).
#          Verify row counts match the unoptimized join.
# TODO 3: Implement the "isolate hot key" pattern: separate Yellow Taxi rows,
#          broadcast the dimension for that subset, union with the rest.

# YOUR CODE HERE

In [ ]:
# ── Topic 2 — Benchmark ──────────────────────────────────────────────────────
import math

def bench(label, df, n=2):
    times = []
    for _ in range(n):
        t = time.time()
        df.count()
        times.append(time.time() - t)
    print(f"{label}: avg {sum(times)/n:.2f}s")

# Bad: SMJ on skewed key
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
bad = orders_skewed.join(vehicle_types, on="vehicle_type_id")
bench("BAD: SMJ + skew", bad)

# Good: AQE + broadcast (dimension is tiny)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
good = orders_skewed.join(vehicle_types, on="vehicle_type_id")
bench("GOOD: AQE broadcast", good)

### Production Best Practices — Data Skew Handling

1. **Always check key distribution before optimizing** — run `groupBy(joinKey).count().orderBy(desc)` on samples; skew ratio > 5× warrants action.
2. **Prefer AQE skewJoin for unknown skew patterns** — it handles skew automatically without code changes; set `skewedPartitionThresholdInBytes` to a value reachable in your cluster (default 256 MB).
3. **Use salting only when AQE is unavailable or threshold is unreachable** — salting multiplies dimension table size by the salt factor; budget memory accordingly.
4. **Isolate hot keys for business-critical skew** — when one key (e.g., "Unknown" customer) represents 50%+ of rows, filter it out, handle separately, then union.
5. **Monitor skew after every major data change** — a dataset that was balanced at launch can become skewed as usage patterns evolve.
6. **Avoid GROUP BY on high-cardinality + skewed columns together** — partial aggregation (`HashAggregate` local) reduces shuffle data; ensure `spark.sql.aggregate.useObjectHashAggregateExec` is true.
7. **Salt key must be deterministic per-row, not global** — use `(rand() * N).cast("int")` on the fact side; `explode(array(range(N)))` on the dimension side.

### Follow-up Questions — Topic 2

1. AQE skewJoin splits the hot partition into sub-tasks. How does Spark ensure the matching rows from the other side are correctly replicated to each sub-task?
2. You salt with 10 buckets but the hot key still produces a straggler. What went wrong, and how do you diagnose it?
3. Explain why salting does not work with outer joins without additional logic.
4. After salting, your final result has duplicate rows. What is the root cause?
5. When is speculative execution useful for skew, and when is it harmful?

In [ ]:
# ── Topic 2 — Full Optimized Solution ───────────────────────────────────────
# OPTIMIZED: Strategy 1 — AQE broadcast (dimension fits in memory)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(50 * 1024 * 1024))

result_aqe = orders_skewed.join(vehicle_types, on="vehicle_type_id", how="inner")
print("=== Strategy 1: AQE + Broadcast ===")
result_aqe.explain()

# OPTIMIZED: Strategy 2 — Salting (for when broadcast is not possible)
SALT_N = 4
# Add random salt to fact table
orders_salted = orders_skewed.withColumn("salt", (rand(seed=99) * SALT_N).cast("int"))
# Replicate dimension table for each salt bucket
from pyspark.sql.functions import array, explode
dim_exploded = (vehicle_types
    .withColumn("salt", explode(array([lit(i) for i in range(SALT_N)]))))
# Join on both keys; drop salt after join
result_salt = (orders_salted
    .join(dim_exploded, on=["vehicle_type_id", "salt"], how="inner")
    .drop("salt"))
print("=== Strategy 2: Salting (4 buckets) ===")
result_salt.explain()
print("Row count:", result_salt.count())

## Topic 3 — Small Files Problem

### Business Scenario

| Aspect | Detail |
|--------|--------|
| Symptom | Parquet read stage launches 50,000 tasks for a 10 GB dataset |
| Impact | Task scheduling overhead dominates; driver OOM on file listing |
| Goal | Reduce task count to < 500 while keeping data readable |

**Dataset:** Simulated streaming output — 10,000 tiny Parquet files, one record each.

In [ ]:
# ── Topic 3 — Data Setup ──────────────────────────────────────────────────────
import tempfile, shutil

# Simulate small-files scenario: write many tiny partitions
small_files_dir = tempfile.mkdtemp(prefix="spark_small_files_")
coalesced_dir   = tempfile.mkdtemp(prefix="spark_coalesced_")

# Write 500 partitions (simulates real small-files scenario without writing 50k actual files)
(spark.range(50_000)
    .withColumn("vehicle_type", (col("id") % 10).cast("string"))
    .withColumn("fare", (rand(seed=20) * 50).cast("double"))
    .repartition(500)  # many small partitions
    .write.mode("overwrite").parquet(small_files_dir))

small_df = spark.read.parquet(small_files_dir)
print(f"Small-files read: {small_df.rdd.getNumPartitions()} partitions")
print(f"Row count: {small_df.count()}")

### Problem Statement — File Listing Overhead and Task Explosion

**Symptoms in production:**
- Driver log: `Listing directory: s3a://bucket/data/` taking 5+ minutes
- `FileScanRDD` shows 50,000 partitions in Spark UI
- Jobs tab: "Tasks: 50,000 / 50,000" for a simple aggregation
- GC pressure on driver due to holding 50,000 `FileStatus` objects in memory

**Root cause:** Default `spark.sql.files.maxPartitionBytes = 128 MB` creates one task per file when files are smaller than the threshold. A pipeline writing one record per task produces one file per task.

In [ ]:
# ── Topic 3 — BAD PATTERN ────────────────────────────────────────────────────
# BAD: Read small-files dataset with default settings — many tasks, no coalescing
spark.conf.set("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))  # 128 MB default
spark.conf.set("spark.sql.adaptive.enabled", "false")  # disable AQE coalescing

bad_df = spark.read.parquet(small_files_dir)
print("=== BAD: default read of small files ===")
print(f"Partitions: {bad_df.rdd.getNumPartitions()}")
bad_df.explain()
# Observe: FileScan with many tiny input partitions

### Interview Questions — Topic 3: Small Files Problem

1. Explain why the small files problem affects both the **write path** (producing small files) and the **read path** (consuming small files). What exactly happens at each end?

2. `spark.sql.files.maxPartitionBytes` controls how Spark groups files into partitions. If you have 1,000 files of 1 MB each, how many tasks does Spark create with the default 128 MB setting? What if you set it to 1 MB?

3. What is the difference between `coalesce(N)` and `repartition(N)` for the write path? When does each one cause a full shuffle, and when doesn't it?

4. You're writing a partitioned dataset with `partitionBy("year","month")`. After each daily load, the partition count doubles. Describe the lifecycle of the small files problem over 1 year.

5. Delta Lake's `OPTIMIZE` command compacts small files. Without Delta, how would you implement a file compaction job in PySpark? What are the concurrency risks?

6. `spark.sql.adaptive.coalescePartitions.enabled` merges post-shuffle partitions. Does it also merge input file partitions on read? Why or why not?

7. What is `openCostInBytes` in Spark's file scan? How does it affect task assignment for small files?

8. Explain the difference between `spark.sql.files.maxPartitionBytes`, `spark.sql.files.openCostInBytes`, and `spark.default.parallelism` in the context of reading many small files.

### Expected Logical Plan Discussion — Small Files

**Reading small files (bad):**
```
Relation [id, vehicle_type, fare] parquet
```
Catalyst sees this as a single logical relation regardless of file count. The small-files problem is a **physical planning** issue — Catalyst does not see individual files.

**After coalesce:**
```
Repartition 50, false          ← false = coalesce (no full shuffle)
+- Relation [id, vehicle_type, fare] parquet
```

Catalyst's `CollapseRepartition` rule merges adjacent repartition nodes; it will NOT merge across shuffle boundaries.

### Expected Physical Plan Discussion — Small Files

**Bad plan (many tiny tasks):**
```
FileScan parquet [id,vehicle_type,fare]
  Format: Parquet, Location: InMemoryFileIndex[file://.../small_files/]
  PushedFilters: []
  ReadSchema: struct<id:long,vehicle_type:string,fare:double>
  PartitionFilters: [], DataFilters: []
  numFiles: 500, totalSize: X MB
```
Task count = ceil(totalSize / maxPartitionBytes) — but each file is below threshold, so each becomes its own task.

**Good plan (coalesced on read):**
```
FileScan parquet [id,vehicle_type,fare]
  numFiles: 500, totalSize: X MB
→ Spark groups files: totalSize / maxPartitionBytes tasks (much fewer)
```

With AQE + `coalescePartitions`, a post-shuffle `AQEShuffleRead coalesced` node further reduces partition count.

### Spark UI Investigation Guide — Small Files Problem

**Step 1 — Jobs tab → task count**
- A simple `count()` showing 500+ tasks on a small dataset = small files
- Look at "Input Size" column: tasks reading < 1 MB each = small files confirmed

**Step 2 — Stages tab → FileScan stage**
- Click the FileScan stage → "Input Size / Records" column per task
- Median task input < 1 MB with 128 MB `maxPartitionBytes` = below coalescing threshold

**Step 3 — SQL tab → FileScan node metrics**
- Expand FileScan: `numFiles`, `scanTime`, `bytesRead`
- High `scanTime` relative to `bytesRead` = file listing overhead

**Step 4 — Driver logs (if accessible)**
- Search for `Listing directory:` — if repeated thousands of times, S3/HDFS listing is the bottleneck
- `InMemoryFileIndex` size in driver GC logs confirms driver memory pressure

**Step 5 — After fix**
- Rerun: task count should drop to totalSize / maxPartitionBytes
- `Input Size / Records` per task should be close to `maxPartitionBytes`

In [ ]:
# ── Topic 3 — Optimization Exercise (TODO) ───────────────────────────────────
# TODO 1: Read the small-files dataset and use coalesce(10) before aggregation.
#          Print partition count before and after coalesce.
# TODO 2: Set spark.sql.files.maxPartitionBytes to 32 MB and re-read.
#          Compare task count to the default 128 MB setting.
# TODO 3: Re-enable AQE coalescePartitions and observe how post-shuffle partitions shrink.

# YOUR CODE HERE
spark.conf.set("spark.sql.adaptive.enabled", "true")
# ...


In [ ]:
# ── Topic 3 — Benchmark ──────────────────────────────────────────────────────
def bench(label, df, n=2):
    times = []
    for _ in range(n):
        t = time.time()
        df.groupBy("vehicle_type").agg(spark_sum("fare")).collect()
        times.append(time.time() - t)
    print(f"{label}: avg {sum(times)/n:.2f}s | partitions={df.rdd.getNumPartitions()}")

spark.conf.set("spark.sql.adaptive.enabled", "false")
bad_df = spark.read.parquet(small_files_dir)
bench("BAD: 500 small-file partitions", bad_df)

# Good: coalesce before processing
good_df = spark.read.parquet(small_files_dir).coalesce(8)
bench("GOOD: coalesce(8)", good_df)

# Cleanup
shutil.rmtree(small_files_dir, ignore_errors=True)
shutil.rmtree(coalesced_dir, ignore_errors=True)

### Production Best Practices — Small Files Problem

1. **Coalesce before partitioned writes** — `df.repartition(col("year"), col("month")).write.partitionBy(...)` groups data per partition column before writing, reducing file count.
2. **Set `maxPartitionBytes` to match your cluster's task throughput** — 256–512 MB per task is typical for CPU-bound jobs; 64–128 MB for I/O-bound scans.
3. **Use `spark.sql.files.openCostInBytes`** to account for S3/GCS open cost when grouping many small files into fewer tasks.
4. **Schedule periodic compaction jobs** — read, coalesce, overwrite partitions that have accumulated > 1,000 files.
5. **Prevent small files at the source** — configure streaming micro-batch intervals and trigger thresholds to write larger files (> 64 MB each).
6. **Use `INSERT OVERWRITE` in partition granularity** — overwrite only the partitions you're changing, not the whole table.
7. **Monitor file count in metadata** — alert when any partition exceeds 1,000 files; act before query performance degrades.

### Follow-up Questions — Topic 3

1. You run a compaction job (`read → coalesce → overwrite`) on an S3 partition that is being actively written by a streaming job. What data corruption risk exists, and how do you mitigate it?
2. Delta Lake's `OPTIMIZE` uses bin-packing. How does it decide which files to compact together, and what's the target file size?
3. Explain `spark.sql.adaptive.coalescePartitions.minPartitionNum`. When would you increase it above the default?
4. A job reads 1,000 files of 200 MB each. How does `maxPartitionBytes=128MB` affect task count? Is this a small-files problem?
5. What are the trade-offs between compacting in Spark vs. using a cloud-native compaction service (e.g., AWS Glue Compact)?

In [ ]:
# ── Topic 3 — Full Optimized Solution ───────────────────────────────────────
import tempfile, shutil

# Write fresh small-files dataset for solution demo
sol_small_dir  = tempfile.mkdtemp(prefix="spark_sol_small_")
sol_compact_dir = tempfile.mkdtemp(prefix="spark_sol_compact_")

(spark.range(50_000)
    .withColumn("vehicle_type", (col("id") % 10).cast("string"))
    .withColumn("fare", (rand(seed=20) * 50).cast("double"))
    .repartition(500).write.mode("overwrite").parquet(sol_small_dir))

# OPTIMIZED: Read and immediately coalesce to reduce task count
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# OPTIMIZED: Increase maxPartitionBytes so files are grouped more aggressively
spark.conf.set("spark.sql.files.maxPartitionBytes", str(256 * 1024 * 1024))

optimized_df = (spark.read.parquet(sol_small_dir)
    .coalesce(8))  # OPTIMIZED: explicitly merge small partitions
print(f"OPTIMIZED partitions: {optimized_df.rdd.getNumPartitions()}")

# OPTIMIZED: Write compacted output — far fewer, larger files
(optimized_df
    .repartition(4, col("vehicle_type"))  # OPTIMIZED: sort by partition key for pruning
    .write.mode("overwrite")
    .partitionBy("vehicle_type")
    .parquet(sol_compact_dir))

# Verify compacted read
compact_read = spark.read.parquet(sol_compact_dir)
print(f"Compact partitions on read: {compact_read.rdd.getNumPartitions()}")
compact_read.groupBy("vehicle_type").agg(spark_sum("fare").alias("total")).show()

shutil.rmtree(sol_small_dir, ignore_errors=True)
shutil.rmtree(sol_compact_dir, ignore_errors=True)

## Topic 4 — Executor Memory / OOM Issues

### Business Scenario

| Aspect | Detail |
|--------|--------|
| Symptom | `java.lang.OutOfMemoryError: GC overhead limit exceeded` in Stage 5 |
| Impact | Job fails after 2 hours; no partial output |
| Goal | Complete the job without OOM by tuning memory fractions and windowing strategy |

**Dataset:** Window function over a large synthetic dataset partitioned poorly (unbounded window).

In [ ]:
# ── Topic 4 — Data Setup ──────────────────────────────────────────────────────
# Large synthetic dataset for window function OOM scenario
trip_data = (spark.range(300_000)
    .withColumn("driver_id", (col("id") % 100).cast("long"))
    .withColumn("fare",      (rand(seed=30) * 60 + 5).cast("double"))
    .withColumn("trip_date", expr("date_add(date'2024-01-01', cast(id % 365 as int))")))

print(f"trip_data rows: {trip_data.count()}, partitions: {trip_data.rdd.getNumPartitions()}")
trip_data.printSchema()

### Problem Statement — Unbounded Window + Memory Pressure

**Root causes of executor OOM:**
1. **Unbounded window** — `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` buffers ALL rows in the partition in memory before computing any result
2. **Too few partitions** — one executor holds the entire dataset in its window buffer
3. **`spark.memory.fraction` too low** — execution memory (for joins, sorts, windows) competes with storage memory (cache)
4. **Python UDFs** — row-by-row serialization; each row crosses the JVM↔Python boundary, doubling memory usage

**OOM sequence:** buffer fills → GC triggered → GC overhead > 98% → JVM throws OOM.

In [ ]:
# ── Topic 4 — BAD PATTERN ────────────────────────────────────────────────────
# BAD: Unbounded window over entire dataset — buffers all rows in memory
from pyspark.sql.window import Window

# BAD: No partitionBy → entire dataset is one partition → one executor holds all data
bad_window = Window.orderBy("trip_date")  # unbounded, no partitionBy

bad_result = trip_data.withColumn(
    "cumulative_fare",
    spark_sum("fare").over(bad_window)  # running total over ALL rows
)
print("=== BAD: unbounded window, no partitionBy ===")
bad_result.explain()
# This will serialize all 300k rows into one executor's memory
# On small local data it works; on real clusters it causes OOM
print("(plan shown; execution skipped to avoid OOM on local mode)")

### Interview Questions — Topic 4: Executor Memory / OOM

1. Draw the Spark executor memory layout: `spark.executor.memory`, `spark.memory.fraction`, `spark.memory.storageFraction`, overhead, and off-heap. Label which region each operation uses.

2. A job fails with `OutOfMemoryError` in the shuffle sort phase. What configuration changes would you try first, and in what order?

3. Explain the difference between a heap OOM and a GC overhead limit exceeded error. Which is worse from a recovery standpoint?

4. A window function with `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` fails with OOM on a 5 GB partition. You cannot change the window definition. What are your options?

5. What does `spark.sql.windowExec.buffer.in.memory.threshold` control? What happens when the threshold is exceeded?

6. Explain the memory implications of caching a DataFrame with `StorageLevel.MEMORY_AND_DISK` vs `MEMORY_ONLY`. When does the DISK fallback trigger, and at what granularity?

7. A job has 10 executors with 8 GB each. You observe GC time > 30% in the Executors tab. What configuration and code changes would you make?

8. Explain off-heap memory in Spark. What is stored there, and when is it beneficial to enable `spark.memory.offHeap.enabled`?

### Expected Logical Plan Discussion — Executor Memory / OOM

**Unbounded window (bad):**
```
Project [driver_id, fare, trip_date,
         sum(fare) WindowingExpr(unboundedpreceding$(), unboundedfollowing$()) AS cumulative_fare]
+- Relation trip_data
```

**Partitioned window (good):**
```
Project [driver_id, fare, trip_date,
         sum(fare) WindowingExpr(unboundedpreceding$(), currentrow$()) AS running_total]
  Window [sum(fare) over (PARTITION BY driver_id ORDER BY trip_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)]
+- Relation trip_data
```

Key: adding `partitionBy("driver_id")` creates 100 independent windows instead of 1 global window — memory per executor = partition_size / num_partitions.

### Expected Physical Plan Discussion — Executor Memory / OOM

**Bad (unbounded, no partitionBy):**
```
Window [sum(fare) windowspecdefinition(order by trip_date, ROWS BETWEEN...)]
+- Sort [trip_date ASC]
   +- Exchange SinglePartition      ← funnels ALL data to ONE executor!
      +- Scan trip_data
```
`Exchange SinglePartition` is the OOM trigger: it routes every row to a single task.

**Good (partitioned window):**
```
Window [sum(fare) windowspecdefinition(driver_id, order by trip_date, ROWS BETWEEN...)]
+- Sort [driver_id, trip_date ASC]
   +- Exchange hashpartitioning(driver_id, 8)   ← distributed across executors
      +- Scan trip_data
```
Memory per executor ≈ total_data / 8 (or however many partitions).

**Key metric:** `Peak Execution Memory` per task in Spark UI Stages tab — high value = window buffer or sort buffer is large.

### Spark UI Investigation Guide — Executor Memory / OOM

**Step 1 — Executors tab → "GC Time" column**
- Click "Show Additional Metrics"
- GC Time > 10% of task time = memory pressure; > 30% = critical
- Look for executors with high "Peak Memory Usage"

**Step 2 — Stages tab → "Peak Execution Memory" column**
- Requires "Show Additional Metrics" → "Peak Execution Memory"
- One task with 10× higher peak memory = unbounded window or large sort buffer

**Step 3 — Find the OOM stage**
- Jobs tab: look for failed stages (red)
- Click the failed stage → "Failure Reason" column → look for `OutOfMemoryError`
- Check which operator caused it (Window, Sort, HashAggregate)

**Step 4 — SQL tab → Window operator metrics**
- Expand Window node: `numOutputRows`, `spillSize`
- If `spillSize > 0`, the window buffer exceeded `windowExec.buffer.in.memory.threshold`

**Step 5 — Executors tab → "Storage Memory Used" column**
- If storage is consuming > 50% of executor memory, cached DataFrames are crowding out execution memory
- Unpersist unused caches before the OOM stage

In [ ]:
# ── Topic 4 — Optimization Exercise (TODO) ───────────────────────────────────
# TODO 1: Rewrite the window function with partitionBy("driver_id") and
#          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW.
#          Print the plan and confirm Exchange SinglePartition is gone.
# TODO 2: Add a rangeFrame window (RANGE BETWEEN 30 PRECEDING AND CURRENT ROW)
#          and explain how it differs in memory usage from a ROWS frame.
# TODO 3: Set spark.sql.windowExec.buffer.in.memory.threshold to 512 and
#          observe what the explain() plan changes to (spill to disk path).

# YOUR CODE HERE

In [ ]:
# ── Topic 4 — Benchmark ──────────────────────────────────────────────────────
def bench(label, df, n=2):
    times = []
    for _ in range(n):
        t = time.time()
        df.count()
        times.append(time.time() - t)
    print(f"{label}: avg {sum(times)/n:.2f}s")

# Good: partitioned window
good_window = Window.partitionBy("driver_id").orderBy("trip_date").rowsBetween(
    Window.unboundedPreceding, Window.currentRow)
good_df = trip_data.withColumn("running_total", spark_sum("fare").over(good_window))
bench("GOOD: partitioned window (driver_id)", good_df)

# Also show partition count
print(f"Partitions in result: {good_df.rdd.getNumPartitions()}")
good_df.select("driver_id","trip_date","fare","running_total").show(5)

### Production Best Practices — Executor Memory / OOM

1. **Always add `partitionBy` to window functions** — an unpartitioned window funnels all data to one task; partition on the most selective business key.
2. **Size executor memory based on peak task memory, not average** — use Spark UI "Peak Execution Memory" across multiple runs to find the true high-water mark.
3. **Set `spark.memory.fraction=0.8` and `spark.memory.storageFraction=0.3`** for join/window-heavy jobs — gives execution 56% of heap (0.8 × 0.7) vs the default 48%.
4. **Unpersist caches before memory-intensive stages** — `df.unpersist()` immediately frees storage memory for execution; don't wait for GC.
5. **Enable off-heap for shuffle-heavy jobs** — `spark.memory.offHeap.enabled=true` with `offHeap.size=4g` moves shuffle data off the JVM heap, reducing GC pressure.
6. **Watch GC type** — G1GC is default in JDK 9+; for large heaps (> 32 GB) ensure `-XX:+UseG1GC` and tune `G1HeapRegionSize`.
7. **Use `MEMORY_AND_DISK_SER` for large cached DataFrames** — serialized storage uses 2-5× less memory than deserialized at the cost of CPU.

### Follow-up Questions — Topic 4

1. You set executor memory to 8 GB but the job still OOMs at 4 GB of data. Walk through the memory regions to explain how this is possible.
2. What is the difference between `spark.executor.memoryOverhead` and `spark.memory.offHeap.size`? What goes in each region?
3. A job works fine in development (small data) but OOMs in production. Name five reasons this could happen that are NOT simply "the data is bigger."
4. Explain `spark.sql.execution.arrow.pyspark.enabled`. How does Arrow change the memory model for Python UDFs?
5. When should you use `persist(StorageLevel.DISK_ONLY)` instead of `MEMORY_AND_DISK`?

In [ ]:
# ── Topic 4 — Full Optimized Solution ───────────────────────────────────────
# OPTIMIZED: partitionBy restricts each window to driver_id's rows only
# OPTIMIZED: ROWS frame is more memory-efficient than RANGE for dense data
good_window = (Window
    .partitionBy("driver_id")           # OPTIMIZED: 100 independent windows instead of 1
    .orderBy("trip_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow))

result_good = trip_data.withColumn("running_fare", spark_sum("fare").over(good_window))
print("=== OPTIMIZED: partitioned window plan ===")
result_good.explain()
# Confirm: Exchange hashpartitioning(driver_id, ...) replaces Exchange SinglePartition

# OPTIMIZED: Add lag for delta calculation — no extra shuffle needed (same window spec)
lag_window = Window.partitionBy("driver_id").orderBy("trip_date")
result_with_delta = (result_good
    .withColumn("prev_fare", lag("fare", 1).over(lag_window))
    .withColumn("fare_delta", col("fare") - col("prev_fare")))

result_with_delta.select("driver_id","trip_date","fare","running_fare","fare_delta").show(8)
print("Peak execution memory per task visible in Spark UI → Stages tab → Tasks → Peak Execution Memory")

## Topic 5 — Shuffle Spill Analysis

### Business Scenario

| Aspect | Detail |
|--------|--------|
| Symptom | Stage 4 takes 3× longer than expected; Stages tab shows "Spill (Disk): 8 GB" |
| Impact | I/O cost of spill-to-disk triples stage time; network reread during retry doubles it again |
| Goal | Eliminate or minimize shuffle spill by tuning partition count and serialization |

**Dataset:** Multi-way aggregation on large synthetic trip data with 200 default shuffle partitions.

In [ ]:
# ── Topic 5 — Data Setup ──────────────────────────────────────────────────────
# Simulate a large aggregation that would spill with tight memory
large_trips = (spark.range(500_000)
    .withColumn("driver_id",    (col("id") % 5000).cast("long"))
    .withColumn("vehicle_type", (col("id") % 10).cast("string"))
    .withColumn("zone",         (col("id") % 50).cast("string"))
    .withColumn("fare",         (rand(seed=40) * 80).cast("double"))
    .withColumn("distance",     (rand(seed=41) * 25).cast("double")))

print(f"large_trips rows: {large_trips.count()}, partitions: {large_trips.rdd.getNumPartitions()}")

### Problem Statement — Shuffle Spill

**What is shuffle spill?**
- During a shuffle reduce phase, Spark sorts data in memory before writing to disk
- If the sort buffer (`spark.shuffle.spill.compress` region of execution memory) fills up, Spark spills to local disk
- Spilled data must be re-read and merged → 2-3× I/O amplification

**Spill (Memory)** — amount of data deserialized from disk back into memory during the merge
**Spill (Disk)** — amount of data written to disk during spill

**Root causes:**
1. Too many shuffle partitions → each partition holds more data than fits in memory
2. Large working rows (wide schemas, un-pruned columns)
3. Insufficient execution memory (too much storage memory consumed by caches)
4. Slow serialization (Java default vs Kryo)

In [ ]:
# ── Topic 5 — BAD PATTERN ────────────────────────────────────────────────────
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "200")  # BAD: default 200, often too many for small data

# BAD: Grouping on 3 columns with wide schema — large shuffle records
bad_agg = large_trips.groupBy("driver_id", "vehicle_type", "zone").agg(
    spark_sum("fare").alias("total_fare"),
    spark_sum("distance").alias("total_dist"),
    count("*").alias("trip_count")
)
print("=== BAD: 200 shuffle partitions, wide schema ===")
bad_agg.explain()
# In production: Stages tab would show Spill (Disk) > 0 for this pattern

### Interview Questions — Topic 5: Shuffle Spill Analysis

1. A stage shows `Spill (Memory): 2 GB` and `Spill (Disk): 500 MB`. Which is larger in terms of I/O cost, and why?

2. Explain the shuffle read/write lifecycle: from map output → disk → network → reducer → sort buffer → spill. At which exact steps does spill occur?

3. Increasing `spark.sql.shuffle.partitions` reduces spill per partition but increases scheduling overhead. How do you find the optimal value empirically?

4. What does `spark.shuffle.spill.compress=true` do? Does compression reduce spill I/O or eliminate it?

5. Explain how Kryo serialization reduces shuffle spill compared to Java serialization. What is the typical size reduction factor?

6. A stage has 200 shuffle partitions, each with 50 MB of data (10 GB total). Executor memory is 4 GB with `memory.fraction=0.6`. Will this stage spill? Show your calculation.

7. `spark.shuffle.file.buffer` controls the in-memory buffer for shuffle write. How does increasing it help, and what is the risk of setting it too high?

8. How does AQE coalesce partitions interact with shuffle spill? Can AQE eliminate the root cause of spill?

### Expected Physical Plan Discussion — Shuffle Spill

**Spill-prone plan (200 partitions, wide schema):**
```
HashAggregate(keys=[driver_id, vehicle_type, zone], functions=[sum(fare), sum(distance), count(1)])
+- Exchange hashpartitioning(driver_id, vehicle_type, zone, 200)   ← 200 wide partitions
   +- HashAggregate(keys=[driver_id, vehicle_type, zone], functions=[partial_sum, partial_count])
      +- Scan large_trips [driver_id, vehicle_type, zone, fare, distance]
```

**Good plan (tuned partitions + column pruning):**
```
HashAggregate(...)
+- AQEShuffleRead coalesced   ← AQE merges empty partitions
   +- Exchange hashpartitioning(driver_id, vehicle_type, zone, 8)
      +- HashAggregate(partial)
         +- Project [driver_id, vehicle_type, zone, fare, distance]   ← only needed cols
            +- Scan large_trips
```

`HashAggregate` performs **partial aggregation** locally before shuffle — this reduces shuffle data size dramatically when cardinality is < row count.

### Spark UI Investigation Guide — Shuffle Spill

**Step 1 — Stages tab → Enable Spill columns**
- Click "Show Additional Metrics" at top of Stages tab
- Enable "Spill (Memory)" and "Spill (Disk)" columns
- Any non-zero value = spill occurring

**Step 2 — Stages tab → Shuffle Read Bytes**
- Compare "Shuffle Read Size" vs input data size
- Ratio > 1× = no compression savings; ratio > 3× = likely schema bloat

**Step 3 — Tasks tab within spilling stage**
- Click the spilling stage → Task list
- Sort by "Spill (Disk)" — check if spill is uniform (all tasks spill) or isolated (one task = skew + spill)
- Uniform spill = partition count or memory tuning issue
- Isolated spill = skew problem (fix the skew first)

**Step 4 — SQL tab → HashAggregate node**
- Check `numOutputRows` vs `numInputRows`
- If output ≈ input, partial aggregation is not reducing data → high cardinality key
- High cardinality + fixed memory = spill is likely

**Step 5 — Executors tab → "Disk Used" column**
- Rising disk usage across executors during the shuffle phase confirms spill
- After stage completes, disk usage should return to near-zero

In [ ]:
# ── Topic 5 — Optimization Exercise (TODO) ───────────────────────────────────
# TODO 1: Re-enable AQE. Set shuffle.partitions to 8 (match your data size).
#          Run the aggregation and compare explain() output to the bad plan.
# TODO 2: Register Kryo serializer and re-run. Measure shuffle write bytes.
# TODO 3: Project only the columns needed for the aggregation before the groupBy.
#          Confirm ReadSchema in the plan drops from 5 columns to 3.

# YOUR CODE HERE

In [ ]:
# ── Topic 5 — Benchmark ──────────────────────────────────────────────────────
def bench(label, df, n=2):
    times = []
    for _ in range(n):
        t = time.time()
        df.count()
        times.append(time.time() - t)
    print(f"{label}: avg {sum(times)/n:.2f}s | partitions={df.rdd.getNumPartitions()}")

spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "200")
bad_agg = large_trips.groupBy("driver_id","vehicle_type","zone").agg(spark_sum("fare"), count("*"))
bench("BAD: 200 partitions, no AQE", bad_agg)

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")
good_agg = (large_trips
    .select("driver_id","vehicle_type","zone","fare")  # prune columns
    .groupBy("driver_id","vehicle_type","zone")
    .agg(spark_sum("fare").alias("total_fare"), count("*").alias("trips")))
bench("GOOD: AQE + column pruning", good_agg)

### Production Best Practices — Shuffle Spill Analysis

1. **Set `shuffle.partitions` to totalShuffleBytes / targetPartitionSize** — target 128–256 MB per post-shuffle partition; AQE coalescePartitions handles the remainder.
2. **Enable Kryo serialization for shuffle-heavy jobs** — `spark.serializer=org.apache.spark.serializer.KryoSerializer`; typically 3-5× smaller than Java serialization.
3. **Project before groupBy** — drop all columns not in the groupBy or aggregation to reduce shuffle record size.
4. **Monitor Spill (Disk) as a leading indicator** — spill precedes OOM; treat any spill as a warning, not just a metric to minimize.
5. **Increase `spark.shuffle.file.buffer` to 1 MB** (default 32 KB) for sequential writes to reduce system calls on spill.
6. **Use local disk SSDs for shuffle** — configure `spark.local.dir` to point to fast local NVMe; network reread of spilled data amplifies I/O on remote storage.
7. **Partial aggregation is your first defense** — ensure `spark.sql.aggregate.useObjectHashAggregateExec=true`; `HashAggregate` partial mode reduces shuffle data before it reaches the reducer.

### Follow-up Questions — Topic 5

1. You enabled Kryo but shuffle write bytes did not decrease. Name three reasons why Kryo might not be effective in this case.
2. A job has 0 spill but task durations are still high. What else could be causing slow shuffle stages?
3. Explain `spark.reducer.maxSizeInFlight`. How does it interact with spill and network throughput?
4. `spark.shuffle.compress=true` compresses shuffle files. How does this interact with `spark.shuffle.spill.compress=true`? Are they the same setting?
5. You have a 100-way groupBy that produces 10 billion unique keys. AQE coalesce and tuning do not help. What architectural change would you recommend?

In [ ]:
# ── Topic 5 — Full Optimized Solution ───────────────────────────────────────
# OPTIMIZED: AQE coalesces post-shuffle partitions automatically
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")  # AQE will coalesce down

# OPTIMIZED: Project only required columns → smaller shuffle records → less spill risk
slim_trips = large_trips.select("driver_id", "vehicle_type", "zone", "fare")

# OPTIMIZED: Two-stage aggregation — local partial agg before shuffle
result_opt = (slim_trips
    .groupBy("driver_id", "vehicle_type", "zone")
    .agg(
        spark_sum("fare").alias("total_fare"),
        count("*").alias("trip_count"),
        avg("fare").alias("avg_fare")
    ))
print("=== OPTIMIZED: AQE + column pruning plan ===")
result_opt.explain()
result_opt.orderBy(desc("total_fare")).show(10)
print(f"Output partitions: {result_opt.rdd.getNumPartitions()}")

## Topic 6 — Dynamic Partitioning

### Business Scenario

| Aspect | Detail |
|--------|--------|
| Symptom | Daily ETL overwrites 3 years of Parquet data when only today's partition changed |
| Impact | 3× write amplification; downstream consumers see stale data during overwrite |
| Goal | Overwrite only the affected partition(s) atomically |

**Dataset:** Cabs data partitioned by `VehicleYear`; synthetic daily_trips partitioned by `trip_date`.

In [ ]:
# ── Topic 6 — Data Setup ──────────────────────────────────────────────────────
import tempfile, shutil

dyn_part_dir = tempfile.mkdtemp(prefix="spark_dyn_part_")

# Write initial partitioned dataset
daily_trips = (spark.range(100_000)
    .withColumn("vehicle_type", (col("id") % 5).cast("string"))
    .withColumn("fare", (rand(seed=50) * 60).cast("double"))
    .withColumn("trip_year",  expr("2022 + cast(id % 3 as int)"))
    .withColumn("trip_month", expr("1 + cast(id % 12 as int)")))

(daily_trips.write.mode("overwrite")
    .partitionBy("trip_year", "trip_month")
    .parquet(dyn_part_dir))

print("Initial write complete. Partitions:")
import os
part_dirs = [d for d in os.listdir(dyn_part_dir) if d.startswith("trip_year")]
print(f"  trip_year partitions: {sorted(set(d.split('=')[1].split('/')[0] for d in part_dirs))}")

### Problem Statement — Static Overwrite Deletes All Partitions

**`spark.sql.sources.partitionOverwriteMode = static` (default):**
- When you `write.mode("overwrite").partitionBy(...)`, Spark deletes the **entire output directory** before writing
- Even if only one partition changed, all partitions are deleted and rewritten
- This is non-atomic for readers: there's a window where no data exists

**Symptoms:**
- Write job takes 10× longer than expected (rewriting unchanged partitions)
- Downstream readers get `FileNotFoundException` during the overwrite window
- Storage costs spike from rewriting all partitions daily

In [ ]:
# ── Topic 6 — BAD PATTERN ────────────────────────────────────────────────────
# BAD: Static overwrite mode — deletes entire output dir, then rewrites ALL partitions
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

# Simulate "today's data" update — only 2024 data changed
new_2024_data = (spark.range(10_000)
    .withColumn("vehicle_type", (col("id") % 5).cast("string"))
    .withColumn("fare", (rand(seed=60) * 70).cast("double"))
    .withColumn("trip_year",  lit(2024))
    .withColumn("trip_month", lit(6)))

print("=== BAD: Static overwrite — will delete ALL years, not just 2024 ===")
(new_2024_data.write.mode("overwrite")
    .partitionBy("trip_year", "trip_month")
    .parquet(dyn_part_dir))
# After this write: trip_year=2022 and trip_year=2023 are GONE

### Interview Questions — Topic 6: Dynamic Partitioning

1. Explain the difference between `partitionOverwriteMode=static` and `partitionOverwriteMode=dynamic`. What exact directory operations does each perform?

2. You set `partitionOverwriteMode=dynamic` but the write still overwrites more partitions than expected. What is the most common cause?

3. Explain the file count explosion problem with `partitionBy`. If you have 100 executors writing a dataset partitioned by `(year, month, day)`, how many files can one day's partition contain?

4. What does `repartition(col("trip_year"), col("trip_month"))` do before a `partitionBy` write? When is it essential?

5. Explain `spark.sql.shuffle.partitions` vs `spark.sql.files.maxRecordsPerFile` for controlling output file count in partitioned writes.

6. A `partitionBy` write creates millions of files because `VehicleType` has very high cardinality. Name three strategies to reduce file count while preserving the partitioning scheme.

7. How does Spark determine which executor writes to which partition? Can two executors write to the same partition directory simultaneously?

8. You have a partitioned Parquet table. A consumer queries `WHERE trip_year=2024`. Explain partition pruning: what Spark reads and what it skips at the file-system level.

### Expected Logical Plan Discussion — Dynamic Partitioning

**Static overwrite (bad):**
```
OverwriteByExpression [trip_year, trip_month] partitionOverwriteMode=static
+- Relation new_data [vehicle_type, fare, trip_year, trip_month]
```
Catalyst emits a full DELETE on the root output path before any write begins.

**Dynamic overwrite (good):**
```
OverwriteByExpression [trip_year, trip_month] partitionOverwriteMode=dynamic
+- Relation new_data [vehicle_type, fare, trip_year, trip_month]
```
Catalyst tracks which partition directories were written; only those directories are atomically swapped.

### Expected Physical Plan Discussion — Dynamic Partitioning

**Without repartition (many files):**
```
WriteFilesExec partitionBy=[trip_year, trip_month]
+- Scan new_data [vehicle_type, fare, trip_year, trip_month]
   → Each task writes to each partition it touches → N_tasks × N_partitions files
```

**With repartition (controlled files):**
```
WriteFilesExec partitionBy=[trip_year, trip_month]
+- Sort [trip_year, trip_month ASC]           ← groups rows by partition
   +- Exchange hashpartitioning(trip_year, trip_month, N)
      +- Scan new_data
```
Key: `Exchange hashpartitioning(partitionCols, N)` ensures each task owns complete partitions → 1 file per task per partition, not 1 file per executor per partition.

### Spark UI Investigation Guide — Dynamic Partitioning

**Step 1 — Jobs tab → write job**
- A partitioned write creates a job with 2 stages: sort/shuffle + write
- Look at "Output" column: how many files were written

**Step 2 — Stages tab → write stage**
- "Output Rows" should match input
- "Output Files" = total files written; divide by N partitions to get files per partition
- Target: 1-5 files per partition (< 64 MB each ideally)

**Step 3 — File system check (post-write)**
- Count files per partition: `os.listdir(partition_dir)`
- If file count = num_executors × num_tasks_per_partition → no pre-write repartition
- If file count ≈ N_partitions → repartition is working

**Step 4 — SQL tab → WriteFiles operator**
- Expand WriteFiles: `numFiles`, `numOutputRows`, `numOutputBytes`
- Compare `numOutputBytes / numFiles` to target file size (128 MB ideal)

**Step 5 — Confirm dynamic overwrite worked**
- List surviving partitions: non-written partition directories should still exist
- With static mode: only written partitions survive; all others are deleted

In [ ]:
# ── Topic 6 — Optimization Exercise (TODO) ───────────────────────────────────
# TODO 1: Re-initialize the base dataset (write 2022, 2023, 2024 data).
# TODO 2: Set partitionOverwriteMode=dynamic and write only 2024/June data.
#          Verify 2022 and 2023 partitions still exist afterward.
# TODO 3: Use repartition(col("trip_year"), col("trip_month")) before write.
#          Count files per partition and compare to without repartition.

# YOUR CODE HERE
import tempfile
new_dir = tempfile.mkdtemp(prefix="spark_dyn_exercise_")
# ...


In [ ]:
# ── Topic 6 — Benchmark ──────────────────────────────────────────────────────
import tempfile, shutil

bench_dir = tempfile.mkdtemp(prefix="spark_dyn_bench_")

# Write base data: 3 years
base = (spark.range(60_000)
    .withColumn("vehicle_type", (col("id") % 5).cast("string"))
    .withColumn("fare", (rand(seed=70) * 60).cast("double"))
    .withColumn("trip_year",  expr("2022 + cast(id % 3 as int)"))
    .withColumn("trip_month", lit(1)))
(base.write.mode("overwrite").partitionBy("trip_year","trip_month").parquet(bench_dir))

update = (spark.range(5_000)
    .withColumn("vehicle_type", (col("id") % 5).cast("string"))
    .withColumn("fare", (rand(seed=71) * 70).cast("double"))
    .withColumn("trip_year", lit(2024)).withColumn("trip_month", lit(7)))

# BAD: static
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
t = time.time()
update.write.mode("overwrite").partitionBy("trip_year","trip_month").parquet(bench_dir)
print(f"STATIC overwrite: {time.time()-t:.2f}s | years remaining: {[d for d in os.listdir(bench_dir) if 'trip_year' in d]}")

# Restore base
(base.write.mode("overwrite").partitionBy("trip_year","trip_month").parquet(bench_dir))

# GOOD: dynamic
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
t = time.time()
update.write.mode("overwrite").partitionBy("trip_year","trip_month").parquet(bench_dir)
years = sorted([d for d in os.listdir(bench_dir) if 'trip_year' in d])
print(f"DYNAMIC overwrite: {time.time()-t:.2f}s | years: {years}")

shutil.rmtree(bench_dir, ignore_errors=True)

### Production Best Practices — Dynamic Partitioning

1. **Default to `partitionOverwriteMode=dynamic`** for all incremental loads — it is safer and faster for partitioned tables where only a subset changes daily.
2. **Always `repartition(partitionCols)` before a partitioned write** — ensures each output task owns complete partitions, keeping file count under control.
3. **Set `spark.sql.files.maxRecordsPerFile`** to cap file size when repartition by key produces uneven partition sizes.
4. **Partition on low-cardinality, filter-friendly columns** — `year`, `month`, `region`; never partition on a UUID or timestamp.
5. **Monitor files-per-partition over time** — a partition that started with 2 files but now has 2,000 needs compaction.
6. **Use `INSERT OVERWRITE PARTITION` in SparkSQL for atomic single-partition overwrites** — more explicit than DataFrame API and avoids accidental full-table overwrites.
7. **Test `partitionOverwriteMode` in staging first** — the mode is a session-level config; a misconfigured job can silently delete all partitions.

### Follow-up Questions — Topic 6

1. `partitionOverwriteMode=dynamic` is set but a job accidentally deletes all partitions. What could cause this?
2. Explain the atomicity guarantee of Spark's `partitionBy` write. What happens if the executor crashes mid-write?
3. How does `spark.sql.files.maxRecordsPerFile` interact with `repartition` to control output file size?
4. You need to write to a partition that is being read by another job. How do you make this safe?
5. Compare `df.write.partitionBy("year").mode("overwrite")` vs `INSERT OVERWRITE TABLE ... PARTITION(year=2024)` in SparkSQL. Which is safer for concurrent access?

In [ ]:
# ── Topic 6 — Full Optimized Solution ───────────────────────────────────────
import tempfile, shutil

sol_dir = tempfile.mkdtemp(prefix="spark_dyn_sol_")

# Write base: 3 years of data
base_data = (spark.range(90_000)
    .withColumn("vehicle_type", (col("id") % 5).cast("string"))
    .withColumn("fare", (rand(seed=80) * 60).cast("double"))
    .withColumn("trip_year",  expr("2022 + cast(id % 3 as int)"))
    .withColumn("trip_month", expr("1 + cast(id % 12 as int)")))
(base_data.write.mode("overwrite").partitionBy("trip_year","trip_month").parquet(sol_dir))
print(f"Base written. Top-level dirs: {sorted(os.listdir(sol_dir))[:6]}")

# OPTIMIZED: dynamic overwrite — only touches partitions being written
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

update_data = (spark.range(8_000)
    .withColumn("vehicle_type", (col("id") % 5).cast("string"))
    .withColumn("fare", (rand(seed=81) * 75).cast("double"))
    .withColumn("trip_year", lit(2024))
    .withColumn("trip_month", lit(8)))

# OPTIMIZED: repartition by partition columns before write → 1 task per partition
(update_data
    .repartition(col("trip_year"), col("trip_month"))  # OPTIMIZED: controlled file count
    .write.mode("overwrite")
    .partitionBy("trip_year", "trip_month")
    .parquet(sol_dir))

years_after = sorted([d for d in os.listdir(sol_dir) if "trip_year" in d])
print(f"After dynamic overwrite — existing years: {years_after[:6]}")
print("2022 data preserved:", any("trip_year=2022" in d for d in years_after))
print("2024 written:", any("trip_year=2024" in d for d in years_after))

shutil.rmtree(sol_dir, ignore_errors=True)

## Topic 7 — Delta Lake Optimization (OPTIMIZE & COMPACTION)

### Business Scenario

| Aspect | Detail |
|--------|--------|
| Symptom | Delta table scan takes 10 min; `DESCRIBE HISTORY` shows 10,000 files in the latest version |
| Impact | Query latency 50× worse than month 1; file listing alone takes 2 minutes |
| Goal | Compact small files, remove stale data, and configure auto-optimization |

**Note:** Delta Lake requires a Databricks runtime or open-source `delta-spark` package. This notebook simulates the concepts using PySpark Parquet compaction. All Delta-specific commands (`OPTIMIZE`, `VACUUM`, `DESCRIBE HISTORY`) are shown in markdown with exact syntax for reference.

In [ ]:
# ── Topic 7 — Data Setup (PySpark simulation of Delta degradation) ───────────
import tempfile, shutil, os

delta_sim_dir = tempfile.mkdtemp(prefix="spark_delta_sim_")

# Simulate 30 days of daily appends (each append = many tiny files)
print("Simulating 30 daily appends (Delta fragmentation scenario)...")
for day in range(1, 31):
    daily = (spark.range(1_000)
        .withColumn("vehicle_type", (col("id") % 5).cast("string"))
        .withColumn("fare", (rand(seed=day*100) * 60).cast("double"))
        .withColumn("trip_day", lit(day)))
    # Each append creates new files (simulates Delta append-only writes)
    mode = "overwrite" if day == 1 else "append"
    daily.write.mode(mode).parquet(delta_sim_dir)

fragmented_df = spark.read.parquet(delta_sim_dir)
files_before = len([f for f in os.listdir(delta_sim_dir) if f.endswith(".parquet")])
print(f"Files before compaction: {files_before}")
print(f"Total rows: {fragmented_df.count()}")

### Problem Statement — File Accumulation in Delta Tables

**Delta table lifecycle without maintenance:**
1. Day 1: 1 file (initial write)
2. Day 30: 30+ files (one or more per append)
3. Day 365: 1,000+ files (each query must list and open all files)
4. After schema evolution/column additions: even more file versions

**Delta Lake file operations:**
- `OPTIMIZE` — reads all small files in a partition and rewrites them as 1 GB target files
- `VACUUM` — removes files no longer referenced by the transaction log (default retention: 7 days)
- `ZORDER BY` — rewrites files with multi-dimensional clustering for better data skipping

**Auto-optimization configs (Delta):**
```
delta.autoOptimize.optimizeWrite = true   -- bin-pack during write
delta.autoOptimize.autoCompact = true     -- compact after write
```

In [ ]:
# ── Topic 7 — BAD PATTERN (conceptual simulation) ────────────────────────────
# BAD: No compaction — reading all fragmented files
spark.conf.set("spark.sql.adaptive.enabled", "false")

# Simulate what a query planner sees with many small files
bad_read = spark.read.parquet(delta_sim_dir)
print("=== BAD: Reading fragmented files (no compaction) ===")
print(f"Input partitions: {bad_read.rdd.getNumPartitions()}")
bad_read.explain()

# In Delta: DESCRIBE HISTORY would show 30 commits, each with new files
print('''
[Delta equivalent - run on Databricks]:
DESCRIBE HISTORY my_delta_table;
-- Shows 30 WRITE operations, each adding new files

SELECT * FROM my_delta_table WHERE trip_day = 30;
-- Must list ALL 30+ file groups even with partition pruning
''')


### Interview Questions — Topic 7: Delta Lake Optimization

1. Explain the Delta Lake transaction log (`_delta_log`). How does it enable ACID guarantees for concurrent readers and writers?

2. What is the difference between `OPTIMIZE` (bin-packing) and `VACUUM`? Can you run `VACUUM` without first running `OPTIMIZE`?

3. You run `VACUUM RETAIN 0 HOURS`. What are the risks, and why does Delta require `SET DELTA.retentionDurationCheck.enabled = false` to override the default?

4. `autoOptimize.optimizeWrite=true` consolidates files during the write. What is the trade-off in write latency? When would you disable it?

5. Explain time travel in Delta Lake. A user queries `VERSION AS OF 10` but you've run `VACUUM`. What happens?

6. How does Delta's data skipping work? What metadata does Delta maintain per file to enable it?

7. You have a Delta table with 10,000 files, each 1 MB. After `OPTIMIZE`, how many files would you expect? What determines the target file size?

8. Explain the difference between `OPTIMIZE` with `ZORDER BY (col1, col2)` and `OPTIMIZE` without ZORDER. What additional work does ZORDER do, and what is its cost?

### Expected Logical Plan Discussion — Delta Lake Optimization

**Delta scan with data skipping (optimized table):**
```
Relation delta_table [vehicle_type, fare, trip_day]
→ Delta log provides per-file min/max stats
→ Catalyst pushes predicates to Delta's DataSkippingReader
→ Files where max(trip_day) < 30 OR min(trip_day) > 30 are skipped entirely
```

**Key Catalyst rule for Delta:** `OptimizeMetadataOnlyQuery` — if the query only touches metadata (e.g., `COUNT(*)`, `MAX`), Delta returns the answer from the transaction log without reading any data files.

**Without OPTIMIZE (fragmented):**
- Each small file has narrow min/max ranges → skipping is less effective
- File listing itself takes O(numFiles) API calls to S3/ADLS

### Expected Physical Plan Discussion — Delta Lake Optimization

**Before OPTIMIZE (fragmented files):**
```
FileScan delta [vehicle_type, fare, trip_day]
  Format: DeltaParquetFileFormat
  Location: DeltaFileIndex[file:///delta_table/] (10,000 files, 10 GB)
  PartitionFilters: [trip_day = 30]
  DataFilters: []
  SelectedFiles: 10,000 (data skipping works per file only; many files have overlapping ranges)
```

**After OPTIMIZE (compacted):**
```
FileScan delta [vehicle_type, fare, trip_day]
  Location: DeltaFileIndex[file:///delta_table/] (10 files, 10 GB)
  PartitionFilters: [trip_day = 30]
  SelectedFiles: 1   ← data skipping eliminates 9 of 10 compacted files
```

**Key metric to watch:** `numFiles` in the FileScan node. Fewer files = faster listing + better data skipping.

### Spark UI Investigation Guide — Delta Lake Optimization

**Step 1 — SQL tab → FileScan metrics**
- Expand FileScan node: `numFiles`, `scanTime`, `bytesRead`
- High `scanTime` relative to `bytesRead` = file listing overhead (many small files)
- After OPTIMIZE: `numFiles` drops; `scanTime` drops proportionally

**Step 2 — Jobs tab → read job stages**
- Before OPTIMIZE: Stage 1 (FileScan) has 10,000 tasks
- After OPTIMIZE: Stage 1 has ~10 tasks (one per compacted file)

**Step 3 — Delta-specific: `DESCRIBE DETAIL`**
```sql
DESCRIBE DETAIL my_delta_table;
-- numFiles: current active file count
-- sizeInBytes: total table size
```

**Step 4 — Delta: `DESCRIBE HISTORY` for OPTIMIZE runs**
```sql
DESCRIBE HISTORY my_delta_table LIMIT 10;
-- Look for "OPTIMIZE" operations in the "operation" column
-- "operationMetrics" shows numFilesAdded, numFilesRemoved, numOutputBytes
```

**Step 5 — After VACUUM**
- `numFiles` in `DESCRIBE DETAIL` should match post-OPTIMIZE count
- Verify time-travel still works for versions within retention window

In [ ]:
# ── Topic 7 — Optimization Exercise (TODO) ───────────────────────────────────
# TODO 1: Compact the fragmented Parquet simulation using coalesce().
#          Count files before and after.
# TODO 2: Write the Delta equivalent commands you would run on a Databricks cluster.
# TODO 3: Explain in comments when you would use OPTIMIZE vs manual compaction in PySpark.

# YOUR CODE HERE
# PySpark compaction simulation:
# compacted = spark.read.parquet(delta_sim_dir).coalesce(???)
# compacted.write.mode("overwrite").parquet(???)


In [ ]:
# ── Topic 7 — Benchmark ──────────────────────────────────────────────────────
import tempfile, shutil

compact_dir = tempfile.mkdtemp(prefix="spark_compact_dir_")

def bench(label, df, n=2):
    times = []
    for _ in range(n):
        t = time.time()
        df.groupBy("vehicle_type").agg(spark_sum("fare")).collect()
        times.append(time.time() - t)
    print(f"{label}: avg {sum(times)/n:.2f}s | partitions={df.rdd.getNumPartitions()}")

# Fragmented
frag = spark.read.parquet(delta_sim_dir)
bench("FRAGMENTED (30 file groups)", frag)

# Compacted (simulate OPTIMIZE)
(frag.coalesce(2).write.mode("overwrite").parquet(compact_dir))
comp = spark.read.parquet(compact_dir)
bench("COMPACTED (coalesce to 2 files)", comp)

shutil.rmtree(delta_sim_dir, ignore_errors=True)
shutil.rmtree(compact_dir, ignore_errors=True)

### Production Best Practices — Delta Lake Optimization

1. **Schedule `OPTIMIZE` daily or weekly** depending on write frequency; use `WHERE` clause to target only recently written partitions.
2. **Set `delta.autoOptimize.optimizeWrite=true` for streaming sinks** — prevents small-file accumulation without manual compaction jobs.
3. **Never run `VACUUM` with retention < 7 days** unless you have confirmed no time-travel queries or concurrent readers depend on older versions.
4. **Use `OPTIMIZE ZORDER BY (col1, col2)`** when queries consistently filter on multiple columns — provides 10-100× file skip improvement over unordered OPTIMIZE.
5. **Monitor `numFiles` in `DESCRIBE DETAIL`** as a KPI; alert if it exceeds 10,000 for a partition.
6. **Combine `autoCompact` with a weekly full OPTIMIZE** — autoCompact handles day-to-day accumulation; weekly OPTIMIZE ensures large files are properly sized.
7. **Partition Delta tables before OPTIMIZE** — OPTIMIZE is partition-aware; compaction runs within partition boundaries, preserving partition pruning.

### Follow-up Questions — Topic 7

1. You run `OPTIMIZE` on a 1 TB Delta table. The job takes 4 hours and rewrites all 1 TB. Is this expected? How can you speed it up?
2. Explain how Delta's transaction log enables concurrent writes without locks. What happens when two writers commit at the same time?
3. After enabling `autoOptimize.optimizeWrite`, write latency increased by 30%. How do you diagnose and fix this?
4. What is the difference between Delta Lake's `MERGE INTO` and a standard `INSERT OVERWRITE`? Which produces more small files?
5. Explain Change Data Feed (CDF) in Delta Lake. How does it interact with OPTIMIZE and VACUUM?

In [ ]:
# ── Topic 7 — Full Optimized Solution (PySpark compaction + Delta commands) ──
import tempfile, shutil

sol_frag_dir    = tempfile.mkdtemp(prefix="spark_sol_frag_")
sol_compact_dir = tempfile.mkdtemp(prefix="spark_sol_compact_")

# Re-create fragmented dataset
for day in range(1, 11):
    d = (spark.range(2_000)
        .withColumn("vehicle_type", (col("id") % 5).cast("string"))
        .withColumn("fare", (rand(seed=day*7) * 60).cast("double"))
        .withColumn("trip_day", lit(day)))
    d.write.mode("overwrite" if day==1 else "append").parquet(sol_frag_dir)

before_files = len([f for f in os.listdir(sol_frag_dir) if f.endswith(".parquet")])
print(f"Files before compaction: {before_files}")

# OPTIMIZED: Read, repartition to target file count, write back
fragmented = spark.read.parquet(sol_frag_dir)
# OPTIMIZED: Target ~128 MB per file; for 20 MB dataset, 1-2 files is correct
(fragmented
    .repartition(2, col("vehicle_type"))  # OPTIMIZED: group by filter column for skipping
    .write.mode("overwrite")
    .parquet(sol_compact_dir))

after_files = len([f for f in os.listdir(sol_compact_dir) if f.endswith(".parquet")])
print(f"Files after compaction: {after_files}")

# Read compacted and verify row count preserved
compacted = spark.read.parquet(sol_compact_dir)
print(f"Row count preserved: {compacted.count() == fragmented.count()}")
compacted.groupBy("vehicle_type").agg(spark_sum("fare")).orderBy("vehicle_type").show()

print('''
=== Delta Lake equivalent (run on Databricks) ===
-- Step 1: Compact small files (target 1 GB per file by default)
OPTIMIZE my_delta_table WHERE trip_day >= current_date() - 7;

-- Step 2: Remove stale files (keep 7-day time travel window)
VACUUM my_delta_table RETAIN 168 HOURS;

-- Step 3: Enable auto-optimization on the table
ALTER TABLE my_delta_table SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);
''')

shutil.rmtree(sol_frag_dir, ignore_errors=True)
shutil.rmtree(sol_compact_dir, ignore_errors=True)

## Topic 8 — Z-Ordering (Multi-Dimensional Clustering)

### Business Scenario

| Aspect | Detail |
|--------|--------|
| Symptom | Queries filtering on `VehicleType AND VehicleYear` scan 100% of files despite partition pruning on `VehicleYear` |
| Impact | Query latency 20 sec; should be < 1 sec with proper file clustering |
| Goal | Cluster data so that both `VehicleType` and `VehicleYear` filters skip files |

**Dataset:** `cabs` data written with various clustering strategies and queried with multi-column filters.

In [ ]:
# ── Topic 8 — Data Setup ──────────────────────────────────────────────────────
import tempfile, shutil

z_dir_unclustered = tempfile.mkdtemp(prefix="spark_z_unclustered_")
z_dir_clustered   = tempfile.mkdtemp(prefix="spark_z_clustered_")

# Enrich cabs with numeric fields for clustering demo
cabs_enriched = (cabs
    .withColumn("VehicleYear", col("VehicleYear").cast("int"))
    .filter(col("VehicleYear").isNotNull())
    .withColumn("LicenseType_int", expr("crc32(LicenseType) % 10")))

# Write unclustered (random order)
cabs_enriched.write.mode("overwrite").repartition(8).parquet(z_dir_unclustered)

print(f"Unclustered files: {len([f for f in os.listdir(z_dir_unclustered) if f.endswith('.parquet')])}")
print(f"VehicleYear range: {cabs_enriched.agg({'VehicleYear': 'min'}).collect()[0][0]} - {cabs_enriched.agg({'VehicleYear': 'max'}).collect()[0][0]}")
cabs_enriched.groupBy("VehicleType").count().show()

### Problem Statement — Single-Dimension Partition Cannot Skip Multi-Column Filters

**Standard partitioning limitation:**
- `partitionBy("VehicleType")` enables file skipping on `VehicleType` but NOT on `VehicleYear`
- `partitionBy("VehicleYear")` enables skipping on `VehicleYear` but NOT on `VehicleType`
- Query `WHERE VehicleType='Yellow Taxi' AND VehicleYear=2020`: only ONE dimension is pruned

**Z-order / Morton curve concept:**
- Maps a multi-dimensional space (VehicleType × VehicleYear) to a 1D Z-curve
- Data points nearby in multi-dimensional space end up nearby in the 1D file layout
- Result: a query filtering on ANY subset of the Z-order columns skips files effectively

**PySpark simulation:** `sortWithinPartitions(col1, col2)` is not true Z-order but achieves locality within partitions for both columns, which is measurable with `explain()`.

In [ ]:
# ── Topic 8 — BAD PATTERN ────────────────────────────────────────────────────
# BAD: Random file order — multi-column filters cannot skip files
bad_df = spark.read.parquet(z_dir_unclustered)
print("=== BAD: Unclustered read — no file skipping on VehicleType + VehicleYear ===")
print(f"Input partitions: {bad_df.rdd.getNumPartitions()}")
bad_df.explain()

# Query that would scan ALL files in unclustered layout
result_bad = bad_df.filter(
    (col("VehicleType") == "MEDALLION TAXI") & (col("VehicleYear") == 2019))
print("=== Result of multi-column filter on unclustered data ===")
result_bad.explain()
result_bad.show(5)

### Interview Questions — Topic 8: Z-Ordering

1. Explain the Z-order (Morton) curve mathematically. How does it interleave bits from multiple columns to create a 1D ordering, and why does this preserve multi-dimensional locality?

2. `OPTIMIZE ZORDER BY (col1, col2)` in Delta Lake rewrites files with Z-order clustering. How does Delta use per-file min/max statistics to skip files for queries that filter on these columns?

3. What is the trade-off between adding more columns to a `ZORDER BY` clause? At what point does adding another column stop helping?

4. Compare Z-ordering, range partitioning (`partitionBy`), and bucketing as data organization strategies. When would you choose each?

5. Z-ordering is not native to open-source Spark without Delta. Explain three alternative approaches to achieve multi-column data locality in standard Parquet.

6. After running `OPTIMIZE ZORDER BY (VehicleType, VehicleYear)`, a query `WHERE VehicleYear = 2020` still scans all files. Why might this happen?

7. Explain the concept of "data skipping" in Delta Lake. What metadata does Delta maintain per file, and what types of predicates can it evaluate using that metadata?

8. You have a table with 100 columns. A dashboarding team queries 20 different column combinations. Is Z-ordering a viable solution? What would you recommend instead?

### Expected Logical Plan Discussion — Z-Ordering

**Unclustered (bad):**
```
Filter ((VehicleType = 'MEDALLION TAXI') AND (VehicleYear = 2019))
+- Relation [cols...] parquet
```
Catalyst pushes the filter to `PushedFilters` in the FileScan. But with unclustered data, Parquet row-group statistics overlap — the filter eliminates few row groups.

**After sortWithinPartitions (simulated Z-order):**
```
Filter ((VehicleType = 'MEDALLION TAXI') AND (VehicleYear = 2019))
+- Relation [cols...] parquet (files written with sorted row groups)
```
Same plan, but at the physical file level, row-group min/max for both columns are tight → Parquet reader skips more row groups.

**Delta Z-order (ideal):**
```
Relation delta_table [cols...] DeltaParquetFileFormat
  PartitionFilters: []
  DataFilters: [VehicleType = 'MEDALLION TAXI', VehicleYear = 2019]
  SelectedFiles: 2 of 100   ← Delta file-level skipping using per-file min/max
```

### Expected Physical Plan Discussion — Z-Ordering

**PySpark sortWithinPartitions plan:**
```
Project [VehicleType, VehicleYear, ...]
+- Filter ((VehicleType = MEDALLION TAXI) AND (VehicleYear = 2019))
   +- FileScan parquet [VehicleType, VehicleYear, ...]
        PushedFilters: [IsNotNull(VehicleType), EqualTo(VehicleType,MEDALLION TAXI),
                        IsNotNull(VehicleYear), EqualTo(VehicleYear,2019)]
        ReadSchema: struct<VehicleType:string,VehicleYear:int,...>
```

**Key:** `PushedFilters` are evaluated at the Parquet reader level using row-group statistics. With sorted data, row groups for one file contain only a narrow range of both columns → more row groups pass the filter check → fewer rows scanned.

**Delta ZORDER metric to watch (SQL tab):**
- `filesSkipped` = files eliminated by Delta data skipping
- `rowGroupsSkipped` = Parquet row groups eliminated within opened files
- `bytesRead` = actual data read after skipping

### Spark UI Investigation Guide — Z-Ordering & Data Skipping

**Step 1 — SQL tab → FileScan node (Delta)**
- Expand FileScan: look for `numFilesSkippedByDataSkipping` metric
- Before ZORDER: 0 files skipped; after ZORDER: 70-90% of files skipped for selective queries

**Step 2 — SQL tab → FileScan node (Parquet)**
- `bytesRead` before vs after clustering
- `numOutputRows` should be the same (same result); `bytesRead` should decrease

**Step 3 — Stages tab → FileScan stage**
- Task count: sorted/clustered data → fewer tasks because fewer files pass the filter
- "Input Size / Records" per task: wider variance in clustered data (some tasks read 0 bytes from skipped files)

**Step 4 — Delta: `DESCRIBE DETAIL` after OPTIMIZE ZORDER**
```sql
DESCRIBE DETAIL my_delta_table;
-- numFiles: should decrease (compaction)
-- zOrderedBy: shows which columns were Z-ordered
```

**Step 5 — Query history comparison**
- Run the same multi-column filter query before and after OPTIMIZE ZORDER
- Compare `Scan` stage metrics: bytesRead, taskDuration
- Target: > 50% reduction in bytesRead for selective queries

In [ ]:
# ── Topic 8 — Optimization Exercise (TODO) ───────────────────────────────────
# TODO 1: Write the cabs_enriched dataset using sortWithinPartitions("VehicleType","VehicleYear").
#          Run explain() and compare PushedFilters to the unclustered version.
# TODO 2: Count rows matching VehicleType='MEDALLION TAXI' AND VehicleYear=2019
#          from both clustered and unclustered reads. Verify they match.
# TODO 3: Write the Delta equivalent OPTIMIZE ZORDER BY command you would use on Databricks.

# YOUR CODE HERE

In [ ]:
# ── Topic 8 — Benchmark ──────────────────────────────────────────────────────
def bench_filter(label, df, n=2):
    times = []
    for _ in range(n):
        t = time.time()
        df.filter((col("VehicleType") == "MEDALLION TAXI") & (col("VehicleYear") == 2019)).count()
        times.append(time.time() - t)
    print(f"{label}: avg {sum(times)/n:.2f}s")

# Write clustered version
(cabs_enriched
    .sortWithinPartitions("VehicleType", "VehicleYear")
    .write.mode("overwrite").repartition(8).parquet(z_dir_clustered))

unclustered = spark.read.parquet(z_dir_unclustered)
clustered   = spark.read.parquet(z_dir_clustered)

bench_filter("UNCLUSTERED (random order)", unclustered)
bench_filter("CLUSTERED (sortWithinPartitions)", clustered)

### Production Best Practices — Z-Ordering & Delta Optimization

1. **Choose Z-order columns based on query patterns** — analyze the most frequent filter combinations in production; Z-order by the top 2-3 columns.
2. **Z-order and `partitionBy` are complementary** — partition on date for time-range queries; Z-order within partition on filter columns.
3. **Re-run `OPTIMIZE ZORDER BY` after significant data changes** — Z-order is not preserved by appends; schedule weekly for stable tables, daily for hot tables.
4. **Use `sortWithinPartitions` for non-Delta Parquet** — it tightens Parquet row-group statistics without requiring Delta, achieving 20-50% scan reduction for selective queries.
5. **Limit Z-order to 3-4 columns maximum** — each additional column halves the locality benefit for the previous columns; beyond 4 columns, the Morton curve becomes ineffective.
6. **Monitor `bytesRead` and `rowGroupsSkipped` metrics** — these are the ground-truth indicators of whether Z-ordering is helping for your query pattern.
7. **Combine Z-order with column statistics enabled** — ensure `spark.sql.parquet.filterPushdown=true` and `spark.sql.parquet.recordLevelFilter.enabled=true` for maximum row-group skipping.

### Follow-up Questions — Topic 8

1. You Z-ordered by `(VehicleType, VehicleYear)` but queries filtering only on `VehicleYear` do not improve. Explain why, and what you would do differently.
2. Compare Z-ordering to Hilbert curve ordering. When does Hilbert perform better than Morton (Z-order)?
3. `OPTIMIZE ZORDER BY (col1, col2)` on a 10 TB table is too slow (takes 8 hours). Name three approaches to make it feasible.
4. A query team added a new frequently-queried column after the initial ZORDER. How do you incorporate it without a full table rewrite?
5. Explain how Delta Lake's data skipping relates to Parquet row-group statistics. What metadata does Delta maintain beyond what Parquet provides natively?

In [ ]:
# ── Topic 8 — Full Optimized Solution ───────────────────────────────────────
# OPTIMIZED: sortWithinPartitions on filter columns clusters data for better row-group stats
(cabs_enriched
    .repartition(4, col("VehicleType"))             # OPTIMIZED: hash-partition by primary filter col
    .sortWithinPartitions("VehicleType", "VehicleYear")  # OPTIMIZED: sort within each partition
    .write.mode("overwrite")
    .parquet(z_dir_clustered))

clustered = spark.read.parquet(z_dir_clustered)
print("=== OPTIMIZED: Clustered read with multi-column filter ===")
clustered.filter(
    (col("VehicleType") == "MEDALLION TAXI") & (col("VehicleYear") == 2019)
).explain()
# PushedFilters applied to tightly clustered row groups → fewer rows scanned

result = clustered.filter(
    (col("VehicleType") == "MEDALLION TAXI") & (col("VehicleYear") == 2019)
).select("VehicleType","VehicleYear","Name","LicenseType")
print(f"Result rows: {result.count()}")
result.show(10)

print('''
=== Delta Lake equivalent (run on Databricks) ===
-- Compact AND Z-order in one operation
OPTIMIZE cabs_delta ZORDER BY (VehicleType, VehicleYear);

-- Verify data skipping is working
EXPLAIN EXTENDED
SELECT * FROM cabs_delta
WHERE VehicleType = 'MEDALLION TAXI' AND VehicleYear = 2019;
-- Look for 'numFilesSkippedByDataSkipping' in the output
''')

import shutil
shutil.rmtree(z_dir_unclustered, ignore_errors=True)
shutil.rmtree(z_dir_clustered, ignore_errors=True)
print("All temp directories cleaned up.")
spark.stop()